In [ ]:
# Initialize environment

# uv init --bare
# uv add sympy numpy
# uv add --dev ipykernel

# When cloning the project, run:
# uv sync

In [ ]:
import sympy as sp

In [ ]:
t=sp.symbols('t')
x, theta = sp.symbols('x theta', cls=sp.Function)
x=x(t)
theta=theta(t)

M, m, l, g, beta_m, beta_M = sp.symbols('M m l g beta_m beta_M', real=True, positive=True)
F, T, N = sp.symbols('F T N', real=True)
mu_c, mu_s, v_s, k = sp.symbols('mu_c mu_s v_s k', real=True, positive=True)
F_ext, M_ext = sp.symbols('F_ext M_ext', real=True)

x_dot = x.diff(t)
x_ddot = x_dot.diff(t)

theta_dot = theta.diff(t)
theta_ddot = theta_dot.diff(t)

In [ ]:
eq_M_x = F + (beta_m / l) * theta_dot * sp.cos(theta) + T * sp.sin(theta) - beta_M * x_dot- M * x_ddot + F_ext
eq_M_y = N+T*sp.cos(theta) - M * g + (beta_m / l) * theta_dot * sp.sin(theta)

eq_m_theta = m * g * sp.sin(theta) - m * x_ddot * sp.cos(theta) - (beta_m / l) * theta_dot - m * l * theta_ddot + (M_ext / l)
eq_m_n = T + m * g * sp.cos(theta) + m * x_ddot * sp.sin(theta) - m * l * theta_dot**2

sol = sp.solve([eq_M_x, eq_m_theta, eq_m_n, eq_M_y], (x_ddot, theta_ddot, T, N))

x_ddot_sol = sp.simplify(sol[x_ddot])
theta_ddot_sol = sp.simplify(sol[theta_ddot])
T_sol = sp.simplify(sol[T])
N_sol = sp.simplify(sol[N])

display(x_ddot_sol)
display(theta_ddot_sol)
display(T_sol)
display(N_sol)

In [ ]:
x1, x2, x3, x4 = sp.symbols('x1 x2 x3 x4', real=True)
u = sp.symbols('u', real=True)

subs_dict = {
    x: x1,
    x_dot: x2,
    theta: x3,
    theta_dot: x4,
    F: u
}

f1 = x2
f2 = x_ddot_sol.subs(subs_dict)
f3 = x4
f4 = theta_ddot_sol.subs(subs_dict)

display(sp.Eq(sp.symbols('xdot_1'), f1))
display(sp.Eq(sp.symbols('xdot_2'), f2))
display(sp.Eq(sp.symbols('xdot_3'), f3))
display(sp.Eq(sp.symbols('xdot_4'), f4))

In [ ]:
#Points of equilibrium

F_unforced = [f.subs(u, 0) for f in [f1, f2, f3, f4]]

equilibrium_points = sp.solve(F_unforced, (x1,x2, x3, x4))

print(equilibrium_points)

In [ ]:
X = sp.Matrix([x1, x2, x3, x4])
U = sp.Matrix([u, F_ext, M_ext])
f = sp.Matrix([f1, f2, f3, f4])

h = sp.Matrix([x1, x3])

# A = df/dx, B = df/du, C = dh/dx, D = dh/du
A_sym = sp.simplify(f.jacobian(X))
B_sym = sp.simplify(f.jacobian(U))
C_sym = sp.simplify(h.jacobian(X))
D_sym = sp.simplify(h.jacobian(U))

eq_upright = {x1: 0, x2: 0, x3: 0, x4: 0, u: 0, F_ext: 0, M_ext: 0}

A_up = sp.simplify(A_sym.subs(eq_upright))
B_up = sp.simplify(B_sym.subs(eq_upright))
C_up = sp.simplify(C_sym.subs(eq_upright))
D_up = sp.simplify(D_sym.subs(eq_upright))

display(A_up)
display(B_up)
display(C_up)
display(D_up)


In [ ]:
# Fdt: G(s) = C * (sI - A)^(-1) * B + D

s = sp.symbols('s')
I = sp.eye(4)

G = sp.simplify(C_up * (s * I - A_up).inv() * B_up + D_up)

G_x = sp.cancel(G[0])
G_theta = sp.cancel(G[1])

print("Funzione di Trasferimento G_x(s) = X_M(s)/U(s):")
display(G_x)

print("\nFunzione di Trasferimento G_theta(s) = Theta(s)/U(s):")
display(G_theta)

In [ ]:
from sympy.printing.octave import octave_code

def sympy_to_matlab(name, expr):
    params = [M, m, l, g, beta_M, beta_m, mu_c, mu_s, v_s, k]
    subs_dict = {p: sp.Symbol(f"p.{p.name}") for p in params}
    
    subs_dict.update({
        x: sp.Symbol('x'), x_dot: sp.Symbol('dx'),
        theta: sp.Symbol('th'), theta_dot: sp.Symbol('dth'),
        F: sp.Symbol('u')
    })
    
    expr_subbed = expr.subs(subs_dict)
    
    code = octave_code(expr_subbed, assign_to=name)
    return code

print(sympy_to_matlab('A', A_up))
print(sympy_to_matlab('B_full', B_up))
print(sympy_to_matlab('C', C_up))
print(sympy_to_matlab('D', D_up))

print(sympy_to_matlab('G_x', G_x))
print(sympy_to_matlab('G_theta', G_theta))

In [ ]:
# N_approx = (M + m) * g

T_solutions = sp.solve(eq_m_n, T)[0]
N_solutions = sp.solve(eq_M_y, N)
N_coupled = N_solutions[0].subs(T, T_solutions)
N_approx = N_coupled.subs(x_ddot, 0)
N_approx = sp.simplify(N_approx)

display(N_approx)

In [ ]:
F_friction_cart = (mu_c + (mu_s - mu_c) * sp.exp(-(x_dot / v_s)**2)) * N_approx * sp.tanh(k * x_dot)

eq_M_x_nonlinear = eq_M_x - F_friction_cart

T_expr = sp.solve(eq_m_n, T)[0]
eq_1_decoupled = eq_M_x_nonlinear.subs(T, T_expr)
eq_2_decoupled = eq_m_theta

# eq1 = sp.simplify(-eq_1_decoupled)
# eq2 = sp.simplify(-eq_2_decoupled)

eq1 = -eq_1_decoupled
eq2 = -eq_2_decoupled

M_mat, F_vec = sp.linear_eq_to_matrix([eq1, eq2], [x_ddot, theta_ddot])

# M_mat = sp.simplify(M_mat)
# F_vec = sp.simplify(F_vec)

print(sympy_to_matlab('M_mat', M_mat))

print(sympy_to_matlab('F_vec', F_vec))

In [ ]:
N_sym = sp.symbols('N_approx', real=True)
F_vec_slim = F_vec.subs(N_approx, N_sym)

print(sympy_to_matlab('N_approx', N_approx))
print(sympy_to_matlab('M_mat', M_mat))
print(sympy_to_matlab('F_vec', F_vec_slim))